# LangGraph - Typing

**Type hints** (or type annotations) are a way to specify the expected data types of variables, function parameters, and return values in Python code.

When building AI workflows (passing messages, LLM outputs, etc.), type annotations help make sure every *"step"* receives the data it expects.

They ensure our state, node inputs/outputs, and tool parameters are compatibile and self-documenting.

We define these states (the *"memory"* passed between nodes) as Python `TypedDict`, or Pydantic models.

**`TypedDict`** is a special class from Python's `typing` module. It allows us to define the *expected structure* (keys and value types) of a dictionary **in a type-annotated way**.

A regular dict in Python can have any keys and values:

In [1]:
d = {"name": "Laercio", "age": 37, "extra": [1, 2, 3]}

But with **`TypedDict`**, we can define exactly what keys are allowed and what type of value is expected for each key:

In [2]:
from typing_extensions import TypedDict

class Person(TypedDict):
    name: str
    age: int

p: Person = {"name": "Laercio", "age": 37} # ✅ OK
p2: Person = {"name": "Laercio", "age": "not-a-number"} # ❌ Editor/static checker will warn

In [3]:
p

{'name': 'Laercio', 'age': 37}

LangGraph workflows **pass state between nodes**, and this state is often represented as a dict.

* Without any typing, this would be error-prone — any node could add/remove/change keys at will, and bugs would be hard to trace.

* With `TypedDict`, we can **define the exact shape of the state** — so each step (node, tool, LLM) knows *exactly* what data is available and in what format.

In [4]:
from typing_extensions import TypedDict

class MyState(TypedDict):
    messages: list
    user_id: str

def step1(state: MyState) -> MyState:
    # Do something with state["messages"] and state["user_id"]
    return state

def step2(state: MyState) -> MyState:
    # Do something else with state["messages"] and state["user_id"]
    return state

Additionally, another useful tool in Python is the `lambda` function.

A **lambda function** is an **anonymous, one-line function** defined with the `lambda` keyword. It's used to create small, throwaway functions without giving them a formal name (unlike using `def`).

```python
lambda arguments: expression
```

In [5]:
# Regular function
def add(x, y):
    return x + y

# Lambda version
add_lambda = lambda x, y: x + y

print(add_lambda(2, 3))  # Output: 5

5


LangGraph let us pass any callable as a node — so lambdas are perfect for "one-liner" nodes.

In [6]:
from langgraph.graph import StateGraph, add_messages
from typing import Annotated

class State(TypedDict):
    messages: Annotated[list, add_messages]
    counter: int
    value: int

graph = StateGraph(State)

# Add a lambda node that increments a counter in the state
graph.add_node("increment", lambda state: {"counter": state["counter"] + 1})

In [7]:
# Decide which node to go to next based on a state value
graph.add_conditional_edges(
    "check",
    lambda state: "success" if state["value"] > 0 else "failure"
)